# Experimentation and Causal Inference

Use this notebook when the case study is about treatment impact, uplift, or policy evaluation. It includes:

- A/B test style treatment vs control analysis
- Difference-in-Differences (DiD)
- Propensity score estimation and weighting


## Before You Start

Causal methods require stronger assumptions than ordinary prediction.

Check whether you have:
- A clear treatment indicator
- A clear outcome metric
- Timing information for pre/post analysis
- Confounders that may affect both treatment and outcome


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.proportion import proportions_ztest

pd.set_option('display.max_columns', None)
plt.style.use('ggplot')


In [ ]:
DATA_PATH = Path('data.csv')

# Update these placeholders for your experiment dataset.
TREATMENT_COL = 'added_to_cart'
OUTCOME_COL = 'converted'
DATE_COL = 'signup_date'
UNIT_COL = 'customer_id'
GROUP_COL = 'segment'

df = pd.read_csv(DATA_PATH)
if DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')

display(df.head())


## 1. A/B Test Style Analysis

For binary outcomes, a difference in proportions is often the first simple readout.


In [ ]:
ab_summary = (
    df.groupby(TREATMENT_COL)
    .agg(
        observations=(OUTCOME_COL, 'size'),
        successes=(OUTCOME_COL, 'sum'),
        outcome_rate=(OUTCOME_COL, 'mean'),
    )
)
display(ab_summary)

if set(df[TREATMENT_COL].dropna().unique()) >= {0, 1}:
    successes = ab_summary['successes'].values
    observations = ab_summary['observations'].values
    z_stat, p_value = proportions_ztest(successes, observations)
    print({'z_stat': z_stat, 'p_value': p_value})


## 2. Difference-in-Differences

This requires:
- A treated group and a comparison group
- A pre period and a post period
- Parallel trends assumption to be at least plausible


In [ ]:
INTERVENTION_DATE = pd.Timestamp('2026-01-15')
TREATED_GROUP_VALUE = 'Enterprise'

did_df = df.dropna(subset=[DATE_COL, GROUP_COL, OUTCOME_COL]).copy()
did_df['post_period'] = (did_df[DATE_COL] >= INTERVENTION_DATE).astype(int)
did_df['treated_group'] = (did_df[GROUP_COL] == TREATED_GROUP_VALUE).astype(int)

did_model = smf.ols(
    f"{OUTCOME_COL} ~ treated_group + post_period + treated_group:post_period",
    data=did_df,
).fit(cov_type='HC1')

print(did_model.summary())


In [ ]:
did_plot = (
    did_df.groupby(['treated_group', 'post_period'])[OUTCOME_COL]
    .mean()
    .reset_index()
)

pivoted = did_plot.pivot(index='post_period', columns='treated_group', values=OUTCOME_COL)
pivoted.index = ['Pre', 'Post']
pivoted.columns = ['Control', 'Treatment']
pivoted.plot(marker='o', figsize=(7, 4), title='Difference-in-Differences Visual Check')
plt.ylabel(OUTCOME_COL)
plt.tight_layout()
plt.show()


## 3. Propensity Score Weighting

Use this when treatment is not randomized and you want to adjust for observable confounders.

Suggested covariates:
- Demographics
- Historic behavior
- Acquisition source
- Prior outcomes


In [ ]:
covariate_cols = [
    c for c in df.columns
    if c not in {TREATMENT_COL, OUTCOME_COL, DATE_COL, UNIT_COL}
]

ps_df = df.dropna(subset=[TREATMENT_COL, OUTCOME_COL]).copy()
X_ps = ps_df[covariate_cols]
y_ps = ps_df[TREATMENT_COL]

numeric_features = X_ps.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_ps.columns if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ],
)

propensity_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000)),
])

propensity_model.fit(X_ps, y_ps)
propensity_scores = propensity_model.predict_proba(X_ps)[:, 1]
ps_df['propensity_score'] = propensity_scores
ps_df['iptw'] = np.where(
    ps_df[TREATMENT_COL] == 1,
    1 / ps_df['propensity_score'].clip(0.01, 0.99),
    1 / (1 - ps_df['propensity_score'].clip(0.01, 0.99)),
)

display(ps_df[[TREATMENT_COL, OUTCOME_COL, 'propensity_score', 'iptw']].head())


In [ ]:
weighted_treatment_mean = np.average(
    ps_df.loc[ps_df[TREATMENT_COL] == 1, OUTCOME_COL],
    weights=ps_df.loc[ps_df[TREATMENT_COL] == 1, 'iptw'],
)
weighted_control_mean = np.average(
    ps_df.loc[ps_df[TREATMENT_COL] == 0, OUTCOME_COL],
    weights=ps_df.loc[ps_df[TREATMENT_COL] == 0, 'iptw'],
)

print({
    'weighted_treatment_mean': weighted_treatment_mean,
    'weighted_control_mean': weighted_control_mean,
    'estimated_ate': weighted_treatment_mean - weighted_control_mean,
})


## Assumptions and Limitations

Document these explicitly:

- Was assignment randomized or observational?
- What threats to validity remain?
- For DiD, is the parallel-trends assumption plausible?
- For propensity methods, what important confounders may still be missing?
- What business decision should change if the effect is real?
